# Functional API: Segmentation Made Simple

The functional API provides the simplest interface for segmentation tasks. This notebook demonstrates:

1. Basic `segment_scores()` function
2. Different fitness function examples
3. Comparing results across fitness functions
4. Configuration options

## Setup

In [1]:
import sys
from pathlib import Path

# Add src to path for importing pso_segmentation
sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd

from pso_segmentation import (
    OptimizerConfig,
    example_fitness_r2_only,
    example_fitness_r2_with_all_constraints,
    example_fitness_r2_with_balance_penalty,
    example_fitness_r2_with_monotonic_penalty,
    segment_scores,
)

np.random.seed(42)
print("Libraries imported successfully!")

Libraries imported successfully!


## Generate Sample Data

In [2]:
# Create realistic credit scoring dataset
n_samples = 1500
scores = np.random.beta(a=2, b=5, size=n_samples)
labels = (np.random.rand(n_samples) < scores).astype(int)

print(f"Dataset: {n_samples} customers")
print(f"Default rate: {labels.mean():.1%}")
print(f"Score range: [{scores.min():.3f}, {scores.max():.3f}]")

Dataset: 1500 customers
Default rate: 29.7%
Score range: [0.005, 0.813]


## 1. Basic Usage: R² Only

Maximize explained variance without constraints:

In [3]:
# Simple one-liner segmentation
result_r2 = segment_scores(
    scores, labels, lambda cuts: example_fitness_r2_only(cuts, scores, labels)
)

print("R² Only Segmentation:")
print(f"  R²: {result_r2.r2:.4f}")
print(f"  Segments: {result_r2.n_segments}")
print(f"  PD by segment: {np.round(result_r2.pd_by_segment, 3)}")
print(f"  Segment proportions: {np.round(result_r2.segment_proportions, 3)}")

R² Only Segmentation:
  R²: 0.1614
  Segments: 4
  PD by segment: [0.126 0.292 0.562 0.833]
  Segment proportions: [0.359 0.447 0.149 0.044]


## 2. With Monotonic Constraint

Enforce that default rate increases monotonically:

In [4]:
result_monotonic = segment_scores(
    scores, labels, lambda cuts: example_fitness_r2_with_monotonic_penalty(cuts, scores, labels)
)

print("With Monotonic Constraint:")
print(f"  R²: {result_monotonic.r2:.4f}")
print(f"  PD by segment: {np.round(result_monotonic.pd_by_segment, 3)}")

# Check if monotonic
probdef = result_monotonic.pd_by_segment
is_monotonic = all(probdef[i] <= probdef[i + 1] for i in range(len(probdef) - 1))
print(f"  Is monotonic increasing: {is_monotonic}")

With Monotonic Constraint:
  R²: 0.1607
  PD by segment: [0.126 0.292 0.557 0.833]
  Is monotonic increasing: True


## 3. With Balance Constraint

Encourage roughly equal-sized segments:

In [5]:
result_balanced = segment_scores(
    scores, labels, lambda cuts: example_fitness_r2_with_balance_penalty(cuts, scores, labels)
)

print("With Balance Constraint:")
print(f"  R²: {result_balanced.r2:.4f}")
print(f"  Segment proportions: {np.round(result_balanced.segment_proportions, 3)}")

# Compare proportion variance
prop_var_r2 = np.var(result_r2.segment_proportions)
prop_var_balanced = np.var(result_balanced.segment_proportions)
print("\nVariance in proportions:")
print(f"  R² only: {prop_var_r2:.4f}")
print(f"  Balanced: {prop_var_balanced:.4f}")
print(f"  Reduction: {(prop_var_r2 - prop_var_balanced) / prop_var_r2 * 100:.1f}%")

With Balance Constraint:
  R²: 0.1498
  Segment proportions: [0.226 0.295 0.285 0.193]

Variance in proportions:
  R² only: 0.0259
  Balanced: 0.0018
  Reduction: 93.1%


## 4. With All Constraints

Combine monotonicity and balance:

In [6]:
result_all = segment_scores(
    scores, labels, lambda cuts: example_fitness_r2_with_all_constraints(cuts, scores, labels)
)

print("With All Constraints:")
print(f"  R²: {result_all.r2:.4f}")
print(f"  PD by segment: {np.round(result_all.pd_by_segment, 3)}")
print(f"  Segment proportions: {np.round(result_all.segment_proportions, 3)}")
print(f"  H_inter: {result_all.h_inter:.4f}")
print(f"  H_intra: {result_all.h_intra:.4f}")

With All Constraints:
  R²: 0.1483
  PD by segment: [0.109 0.205 0.318 0.618]
  Segment proportions: [0.226 0.295 0.281 0.197]
  H_inter: 46.4030
  H_intra: 266.5804


## 5. Comparison

Compare all approaches side-by-side:

In [8]:
comparison = pd.DataFrame(
    {
        "Approach": ["R² Only", "Monotonic", "Balanced", "All Constraints"],
        "R²": [result_r2.r2, result_monotonic.r2, result_balanced.r2, result_all.r2],
        "Mono": [
            all(
                result_r2.pd_by_segment[i] <= result_r2.pd_by_segment[i + 1]
                for i in range(result_r2.n_segments - 1)
            ),
            all(
                result_monotonic.pd_by_segment[i] <= result_monotonic.pd_by_segment[i + 1]
                for i in range(result_monotonic.n_segments - 1)
            ),
            all(
                result_balanced.pd_by_segment[i] <= result_balanced.pd_by_segment[i + 1]
                for i in range(result_balanced.n_segments - 1)
            ),
            all(
                result_all.pd_by_segment[i] <= result_all.pd_by_segment[i + 1]
                for i in range(result_all.n_segments - 1)
            ),
        ],
        "Prop_Var": [
            np.var(result_r2.segment_proportions) * 100,
            np.var(result_monotonic.segment_proportions) * 100,
            np.var(result_balanced.segment_proportions) * 100,
            np.var(result_all.segment_proportions) * 100,
        ],
    }
)

print("\nNote:")
print("  - R²: Explained variance (higher is better)")
print("  - Mono: Monotonic increasing default rate")
print("  - Prop_Var: Variance in segment proportions (lower = more balanced)")
print("Comparison Matrix:")
comparison


Note:
  - R²: Explained variance (higher is better)
  - Mono: Monotonic increasing default rate
  - Prop_Var: Variance in segment proportions (lower = more balanced)
Comparison Matrix:


,Approach,R²,Mono,Prop_Var
0,R² Only,0.161422,True,2.586600
1,Monotonic,0.160696,True,2.525022
2,Balanced,0.149844,True,0.177267
3,All Constraints,0.148260,True,0.159667


## 6. Custom Configuration

You can also pass custom PSO configurations to `segment_scores()`:

In [9]:
# Create custom config
config = OptimizerConfig(
    n_segments=3,  # Want exactly 3 segments
    pop_size=100,  # Larger population for better exploration
    max_iter=1000,  # More iterations for convergence
    w=0.7,  # Balanced inertia
    c1=1.5,  # Cognitive parameter
    c2=1.5,  # Social parameter
    seed=42,  # For reproducibility
)

result_custom = segment_scores(
    scores,
    labels,
    lambda cuts: example_fitness_r2_with_all_constraints(cuts, scores, labels),
    config=config,  # Pass the config
)

print("Custom Configuration Result:")
print(f"  R²: {result_custom.r2:.4f}")
print(f"  Segments: {result_custom.n_segments}")
print(f"  PD by segment: {np.round(result_custom.pd_by_segment, 3)}")

Custom Configuration Result:
  R²: 0.1497
  Segments: 3
  PD by segment: [0.126 0.293 0.624]


## Key Takeaways

✅ **Functional API Benefits:**
- Simple one-liner calls
- Perfect for quick experiments
- No boilerplate code needed

✅ **Fitness Function Choices:**
- R² only: Best predictive power
- Monotonic: Ensures interpretability
- Balanced: Easier portfolio management
- All constraints: Production-ready

✅ **Configuration:**
- Can pass `OptimizerConfig` for fine-tuning
- Default config works for most cases
- Seed parameter for reproducibility

## Next Steps

👉 **02_oo_api.ipynb** - For more control, explore the Object-Oriented API

👉 **03_custom_fitness.ipynb** - Create your own fitness functions